<a href="https://colab.research.google.com/github/ramcharan0816/Stress-Detection-using-MachineLearning/blob/main/StressDetection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

###1. Importing necessary libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,classification_report

nltk.download("stopwords")
nltk.download("wordnet")

###2. Explore data

In [ ]:
df=pd.read_csv("stress.csv")
print(df.head())

In [ ]:
df=df[['text','label']]
df.dropna(inplace=True)
df.drop_duplicates(inplace=True)

In [ ]:
print(df.info())
print(df['label'].value_counts())

### 3. Visualize Class Distribution.

In [ ]:
df['label'].value_counts().plot(kind='bar')
plt.title("Stress vs No Stress Count")
plt.xlabel("Label")
plt.ylabel("Count")
plt.show()

###3.1 Displaying words using WordCloud

In [ ]:
from wordcloud import WordCloud
text_data=" ".join(df[df['label']==1]['text'])
wordcloud=WordCloud(width=800,height=500,background_color='white').generate(text_data)

In [ ]:
plt.figure(figsize=(10,5))
plt.imshow(wordcloud)
plt.axis('off')
plt.title( "Stress Related Words")
plt.show()

###4. Text Cleaning Process.

In [10]:
lemm=WordNetLemmatizer()
stop_words=set(stopwords.words("english"))
def clean_text(text):
  text=text.lower()
  text=re.sub(r'http\S+|www\S+', '', text)
  text = re.sub(r'[^a-zA-Z]', ' ', text)

  words=text.split()
  words=[lemm.lemmatize(word) for word in words if word not in stop_words]

  return " ".join(words)
df['clean_text']=df['text'].apply(clean_text)


### 5. Use TF-IDF to convert text into numbers.

In [13]:
vectorizer=TfidfVectorizer()
X=vectorizer.fit_transform(df['clean_text'])
y=df['label']

### 6. Split Dataset into training and testing.

In [14]:
X_train,X_test,y_train,y_test=train_test_split(X,y,random_state=42,test_size=0.2)

### 7. Train the model and predict.

In [15]:
model=LogisticRegression()
model.fit(X_train,y_train)
pred=model.predict(X_test)

In [16]:
print('Accuracy:',accuracy_score(y_test,pred))
print(classification_report(y_test,pred))

Accuracy: 0.7345132743362832
              precision    recall  f1-score   support

           0       0.77      0.67      0.72       285
           1       0.70      0.80      0.75       280

    accuracy                           0.73       565
   macro avg       0.74      0.74      0.73       565
weighted avg       0.74      0.73      0.73       565



###8. Custom Prediction

In [17]:
def predict_stress(text):
  text=clean_text(text)
  vector=vectorizer.transform([text])
  prediction=model.predict(vector)

  if prediction[0]==1:
    return "Stress Detected"
  else:
    return "No Stress"
print(predict_stress("I am too anxious about my exams"))

Stress Detected
